# FahMai RAG Challenge: Advanced Pipeline

This notebook contains a complete flow for high-accuracy RAG:
- **Dense Retrieval** (MiniLM embeddings)
- **Sparse Retrieval** (BM25 with Thai tokenization)
- **Hybrid Retrieval** (Reciprocal Rank Fusion)
- **Reranking** (Cross-Encoder verification)
- **Super Voting** (Multi-model consensus)

In [1]:
from google.colab import files
# file "https://www.kaggle.com/settings"
files.upload() #upload kaggle.json (Legacy API Credentials)

!mkdir -p ~/.kaggle/

# Move the uploaded kaggle.json file to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set secure permissions for the API key file (read-only for owner)
!chmod 600 ~/.kaggle/kaggle.json

! kaggle competitions download -c super-ai-engineer-s-6-fah-mai-rag-challenge-level-1

Saving kaggle.json to kaggle.json
100% 321k/321k [00:00<00:00, 99.7MB/s]



In [2]:
# Load the data from kaggle and upload here.
!unzip super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_

In [3]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
N_QUESTIONS = 100
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"
PREFERRED_LLM_MODEL = "kbtg"
TOP_K = 5

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [4]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv
import os, csv, re, time, requests, numpy as np, pandas as pd
from pathlib import Path
from google.colab import userdata
from collections import Counter
from sentence_transformers import SentenceTransformer, CrossEncoder
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

THAILLM_API_KEY = userdata.get('ThaiLLM_API')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 53.4 MB/s eta 0:00:00


In [5]:
def ask_llm(messages, model=PREFERRED_LLM_MODEL, max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling directly using the model name."""
    url = f"http://thaillm.or.th/api/{model.lower()}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  [{model}] Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except Exception as e:
            wait = 2 ** attempt
            print(f"  [{model}] Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [6]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): <think>
Okay, the user is asking about the water resistance rating of the Samsung Galaxy S3 Ultra, specifically how many ATM it can withstand. First, I need to recall the specifications of the S3 Ultra. I remember that the S3 Ultra is a variant of the Galaxy S3, but I'm not entirely sure about its water resistance. 

Wait, the original Galaxy S3 was not water-resistant. I think the S3 Ultra might have some water resistance, but I need to confirm. Let me check my notes. The S3 Ultra was released in 2013, and at that time, water resistance wasn't a common feature in smartphones. However, some models might have had IP ratings. 

Looking up the specifications, the S3 Ultra has an IP67 rating. IP67 means it's dust-tight and can withstand immersion in water up to 1 meter for 30 minutes. To convert that to ATM, I need to remember that 1 meter of water is approximately 1 ATM. So, 1 meter is 1 ATM, and 30 minutes is the duration. Therefore, the S3 Ultra is rated for 1

### Load Questions

In [7]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions.")

Loaded 100 questions.


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [8]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [9]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

Created 1055 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้


### 1.3 Embedding (BGE-M3)

We are now using `BAAI/bge-m3`, a powerful multilingual model designed for high-performance retrieval.

In [10]:
from sentence_transformers import SentenceTransformer

# Using BGE-M3 as requested
embed_model = SentenceTransformer('BAAI/bge-m3')

# Embed all chunks
chunk_texts = [c['text'] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Chunk embeddings shape: (1055, 1024)


### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [11]:
TOP_K = 5
def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

  Rank 1 (score=0.718) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ือน** ผ่านบัตรเครดิตที่ร่วมรายการ
(หมดเขต 30 มิถุนายน 2569)

ตัวอย่าง: ฿14,990 ÷ 6 เดือน = เพียง ฿2,498.33/เดือน

---

## คำถามที่พบบ่อยเกี่ยวกับสินค้...

  Rank 2 (score=0.702) [products/WK-SW-003_wongkhojon_watch_s3.md]
  ATM ในราคาที่เข้าถึงได้ง่ายกว่า S3 Pro กว่าครึ่ง

ตัวเรือนอะลูมิเนียมอัลลอยด์เคลือบผิวแบบ anodized ให้ความทนทานและน้ำหนักเบาพร้อมกัน หน้าจอ AMOLED 1.7...

  Rank 3 (score=0.688) [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  3 Pro มีฟังก์ชัน ECG (คลื่นไฟฟ้าหัวใจ) ครบ นี่คือความแตกต่างหลักจาก Watch S3 (WK-SW-003) ที่ไม่มี ECG ส่วนฟีเจอร์อื่นอย่าง SpO2, GPS, AMOLED, และกันน้...

  Rank 4 (score=0.685) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  กาศ (Grade 5 Titanium) น้ำหนักเบากว่าสแตนเลสทั่วไปถึง 40% แต่แข็งแกร่งกว่าหลายเท่า ให้ความรู้สึกสวมใส่ที่เบาสบายตลอดทั้งวันแม้ระหว่างออกกำลังกายหนัก

...

  Rank 5 (score=0.684) [products/WK-SW-0

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [12]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = "ตอบคำถามจากข้อมูลที่ให้มา เลือกตัวเลือกที่ถูกต้องที่สุด ตอบเป็น ANSWER: X เท่านั้น"

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context.

    TODO: Design your own prompt format.
    Think about: How should you present the context? The choices?
    What instructions help the LLM pick the right answer?
    """
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    # === CUSTOMIZE THIS PROMPT ===
    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw: <think>
Okay, let's tackle this question. The user is asking about the water resistance of the Watch S3 Ultra in ATM. I need to find the correct answer from the provided chunks.

First, I'll go through each chunk to find relevant information. 

In Chunk 1, there's a mention of S3 Ultra having a 100-meter water resistance with Dive Mode, but that's part of a comparison with S3 Pro. 

Chunk 2 talks about S3 Pro being 5 ATM (50 meters) but doesn't mention the Ultra. 

Chunk 3 compares S3 Ultra and S3 Pro, stating that S3 Ultra is 10 ATM. 

Chunk 4 explicitly says S3 Ultra has 10 ATM water resistance with Dive Mode. 

Chunk 5 also mentions that S3 Ultra has 10 ATM. 

So, the correct answer should be 10 ATM, which is option 5.
</think>

ANSWER: 5
Parsed answer: 5


In [13]:
def compare_four_models(question_obj, models=["typhoon", "openthaigpt", "pathumma", "kbtg"]):
    """Retrieves context once and asks all 4 models the same question."""
    print(f"Question: {question_obj['question']}\n")

    # 1. Retrieve Context (using Hybrid as it's usually most robust)
    # Note: Ensure hybrid_retrieve is defined. Using dense as fallback if needed.
    try:
        idx = hybrid_retrieve(question_obj['question'], chunk_embeddings)
    except NameError:
        idx, _ = dense_retrieve(question_obj['question'], chunk_embeddings)

    retrieved_chunks = [chunks[i] for i in idx]
    prompt = build_rag_prompt(question_obj['question'], question_obj['choices'], retrieved_chunks)

    # 2. Query each model
    for model_name in models:
        print(f"Querying {model_name}...")
        raw_response = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ], model=model_name)

        parsed = parse_answer(raw_response)
        print(f"[{model_name.upper()}] Response: {raw_response}")
        print(f"[{model_name.upper()}] Parsed Answer: {parsed}")
        print("-" * 30)

# Test with Question 1
compare_four_models(questions[0])

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Querying typhoon...
[TYPHOON] Response: ANSWER: 5
[TYPHOON] Parsed Answer: 5
------------------------------
Querying openthaigpt...
[OPENTHAIGPT] Response: <think>
คำถามคือ Watch S3 Ultra กันน้ำได้กี่ ATM จากข้อมูลที่ให้มา ต้องหาข้อมูลเกี่ยวกับความสามารถในการกันน้ำของ Watch S3 Ultra

Chunk 1: ไม่ได้กล่าวถึงความสามารถในการกันน้ำโดยตรง

Chunk 2: กล่าวถึง Watch S3 Pro ที่กันน้ำ 50 เมตร (5 ATM) แต่ไม่ได้กล่าวถึง Watch S3 Ultra

Chunk 3: กล่าวถึง Watch S3 Pro มีฟังก์ชัน ECG และกันน้ำ 50 เมตร (5 ATM) แต่ไม่ได้กล่าวถึง Watch S3 Ultra

Chunk 4: กล่าวถึง Watch S3 Ultra มีมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode

Chunk 5: ไม่ได้กล่าวถึงความสามารถในการกันน้ำโดยตรง

ดังนั้น คำตอบที่ถูกต้องคือ 10 ATM
</think>

ANSWER: 5
[OPENTHAIGPT] Parsed Answer: 5
------------------------------
Querying pathumma...
[PATHUMMA] Response: <think>
Okay, let's tackle this question. The user is asking about the water resistance of the Watch S3 Ultra. The 

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [14]:
def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + Single LLM (kbtg) for speed."""
    predictions = {}

    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ], model=PREFERRED_LLM_MODEL)

        pred = parse_answer(raw)
        final_pred = pred if pred else 1
        predictions[q["id"]] = final_pred

        print(f"Q{q['id']:>3} | {PREFERRED_LLM_MODEL}: {final_pred}")
        time.sleep(0.05)

    return {PREFERRED_LLM_MODEL: predictions}

In [15]:
# Define the dense_retrieve_chunks function, which is needed by run_pipeline
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

# Run the dense retrieval pipeline for kbtg only
print(f"Running Dense Retrieval Pipeline for {PREFERRED_LLM_MODEL}...")
dense_preds_multi = run_pipeline(questions, dense_retrieve_chunks, label='dense', n=N_QUESTIONS)

# Convert to DataFrame
df_multi = pd.DataFrame(dense_preds_multi)
df_multi.index.name = 'id'
df_multi.to_csv('dense_preds_kbtg.csv')

print("Saved dense_preds_kbtg.csv preview:")
display(df_multi.head())

Running Dense Retrieval Pipeline for kbtg...
Q  1 | kbtg: 5
Q  2 | kbtg: 7
Q  3 | kbtg: 2
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q  4 | kbtg: 6
Q  5 | kbtg: 6
Q  6 | kbtg: 8
Q  7 | kbtg: 1
Q  8 | kbtg: 4
Q  9 | kbtg: 4
Q 10 | kbtg: 2
Q 11 | kbtg: 4
Q 12 | kbtg: 1
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 13 | kbtg: 2
Q 14 | kbtg: 4
Q 15 | kbtg: 7
Q 16 | kbtg: 1
Q 17 | kbtg: 8
Q 18 | kbtg: 5
Q 19 | kbtg: 2
Q 20 | kbtg: 2
Q 21 | kbtg: 3
Q 22 | kbtg: 8
Q 23 | kbtg: 3
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 24 | kbtg: 9
Q 25 | kbtg: 5
Q 26 | kbtg: 6
Q 27 | kbtg: 2
Q 28 | kbtg: 7
Q 29 | kbtg: 4
Q 30 | kbtg: 3
Q 31 | kbtg: 5
Q 32 | kbtg: 2
Q 33 | kbtg: 8
Q 34 | kbtg: 5
Q 35 | kbtg: 3
Q 36 | kbtg: 2
Q 37 | kbtg: 8
Q 38 | kbtg: 6
Q 39 | kbtg: 

,kbtg
id,
1,5
2,7
3,2
4,6
5,6


---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [16]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [17]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 1055 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [18]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md

BM25 top-5 sources:
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md


### 2.4 Run All Questions (BM25)

In [19]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

Q  1 | kbtg: 5
Q  2 | kbtg: 7
Q  3 | kbtg: 9
Q  4 | kbtg: 6
Q  5 | kbtg: 6
Q  6 | kbtg: 9
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q  7 | kbtg: 1
Q  8 | kbtg: 1
Q  9 | kbtg: 9
Q 10 | kbtg: 7
Q 11 | kbtg: 4
Q 12 | kbtg: 1
Q 13 | kbtg: 2
Q 14 | kbtg: 9
Q 15 | kbtg: 7
Q 16 | kbtg: 1
Q 17 | kbtg: 8
Q 18 | kbtg: 5
Q 19 | kbtg: 2
Q 20 | kbtg: 8
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 21 | kbtg: 3
Q 22 | kbtg: 6
Q 23 | kbtg: 1
Q 24 | kbtg: 3
Q 25 | kbtg: 5
Q 26 | kbtg: 6
Q 27 | kbtg: 2
Q 28 | kbtg: 7
Q 29 | kbtg: 4
Q 30 | kbtg: 3
Q 31 | kbtg: 4
Q 32 | kbtg: 2
Q 33 | kbtg: 9
Q 34 | kbtg: 5
Q 35 | kbtg: 3
Q 36 | kbtg: 2
Q 37 | kbtg: 8
Q 38 | kbtg: 9
Q 39 | kbtg: 4
Q 40 | kbtg: 9
Q 41 | kbtg: 7
Q 42 | kbtg: 2
Q 43 | kbtg: 4
Q 44 | kbtg: 1
Q 45 | kbtg: 3
Q 46 | kbtg: 1
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://

In [ ]:
# Convert the nested dictionary to a DataFrame
# dense_preds_multi structure: {'model_name': {qid: answer, ...}, ...}
df_bm25_preds = pd.DataFrame(bm25_preds)

# Ensure the index is named 'id' to match the question identifier
df_bm25_preds.index.name = 'id'

# Save to CSV
df_bm25_preds.to_csv('df_bm25_preds.csv')

print("Saved df_bm25_preds.csv with the following preview:")
display(df_bm25_preds.head())

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

### 3.2 Run All Questions (Hybrid)

In [22]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

# Run Hybrid retrieval for kbtg only
print(f"Running Hybrid Retrieval Pipeline for {PREFERRED_LLM_MODEL}...")
hybrid_preds_multi = run_pipeline(questions, hybrid_retrieve_chunks, label='hybrid', n=100)

Running Hybrid Retrieval Pipeline for kbtg...
Q  1 | kbtg: 5
Q  2 | kbtg: 7
Q  3 | kbtg: 2
Q  4 | kbtg: 6
Q  5 | kbtg: 6
Q  6 | kbtg: 8
Q  7 | kbtg: 1
Q  8 | kbtg: 4
Q  9 | kbtg: 4
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 10 | kbtg: 7
Q 11 | kbtg: 4
Q 12 | kbtg: 1
Q 13 | kbtg: 2
Q 14 | kbtg: 3
Q 15 | kbtg: 7
Q 16 | kbtg: 1
Q 17 | kbtg: 8
Q 18 | kbtg: 5
Q 19 | kbtg: 2
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 20 | kbtg: 2
Q 21 | kbtg: 3
Q 22 | kbtg: 8
Q 23 | kbtg: 9
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q 24 | kbtg: 9
Q 25 | kbtg: 5
Q 26 | kbtg: 6
Q 27 | kbtg: 2
Q 28 | kbtg: 7
Q 29 | kbtg: 4
Q 30 | kbtg: 3
Q 31 | kbtg: 4
Q 32 | kbtg: 2
  [kbtg] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/kbtg/v1/chat/compl

In [23]:
from collections import Counter

def run_voting_pipeline(questions, retrieve_fn, models=["typhoon", "openthaigpt", "pathumma", "thalle"], n=N_QUESTIONS):
    """Run retrieval + Multiple LLMs + Majority Voting."""
    results = []

    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        model_answers = {}
        for model_name in models:
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ], model=model_name)

            pred = parse_answer(raw)
            model_answers[model_name] = pred
            time.sleep(0.2)

        # Voting logic
        valid_votes = [v for v in model_answers.values() if v is not None]
        if valid_votes:
            final_vote = Counter(valid_votes).most_common(1)[0][0]
        else:
            final_vote = 1 # Default fallback

        results.append({
            "id": q["id"],
            "votes": model_answers,
            "final_answer": final_vote
        })
        print(f"Q{q['id']:>3}: Votes={model_answers} -> Winner={final_vote}")

    return results

# To execute:
# voting_results = run_voting_pipeline(questions, hybrid_retrieve_chunks, n=5)

### 3.3 Compare All Three Methods

In [24]:
print(f"{'QID':>4} | {'Method':>7} | {PREFERRED_LLM_MODEL.upper():>7}")
print("-" * 30)

for q in questions[:10]:
    qid = q['id']

    # Dense result
    d_val = dense_preds_multi[PREFERRED_LLM_MODEL].get(qid, '-')
    print(f"Q{qid:>3} | {'Dense':>7} | {d_val:>7}")

    # BM25 result
    if 'bm25_preds' in globals():
        b_val = bm25_preds[PREFERRED_LLM_MODEL].get(qid, '-')
        print(f"Q{qid:>3} | {'BM25':>7} | {b_val:>7}")

    # Hybrid result
    if 'hybrid_preds_multi' in globals():
        h_val = hybrid_preds_multi[PREFERRED_LLM_MODEL].get(qid, '-')
        print(f"Q{qid:>3} | {'Hybrid':>7} | {h_val:>7}")

    # Rerank result
    if 'rerank_preds_multi' in globals():
        r_val = rerank_preds_multi[PREFERRED_LLM_MODEL].get(qid, '-')
        print(f"Q{qid:>3} | {'Rerank':>7} | {r_val:>7}")

    print("-" * 30)

 QID |  Method |    KBTG
------------------------------
Q  1 |   Dense |       5
Q  1 |    BM25 |       5
Q  1 |  Hybrid |       5
------------------------------
Q  2 |   Dense |       7
Q  2 |    BM25 |       7
Q  2 |  Hybrid |       7
------------------------------
Q  3 |   Dense |       2
Q  3 |    BM25 |       9
Q  3 |  Hybrid |       2
------------------------------
Q  4 |   Dense |       6
Q  4 |    BM25 |       6
Q  4 |  Hybrid |       6
------------------------------
Q  5 |   Dense |       6
Q  5 |    BM25 |       6
Q  5 |  Hybrid |       6
------------------------------
Q  6 |   Dense |       8
Q  6 |    BM25 |       9
Q  6 |  Hybrid |       8
------------------------------
Q  7 |   Dense |       1
Q  7 |    BM25 |       1
Q  7 |  Hybrid |       1
------------------------------
Q  8 |   Dense |       4
Q  8 |    BM25 |       1
Q  8 |  Hybrid |       4
------------------------------
Q  9 |   Dense |       4
Q  9 |    BM25 |       9
Q  9 |  Hybrid |       4
---------------------

In [25]:
def get_final_answer(qid, results_dict):
    # Simple lookup for the preferred model result
    return results_dict.get(PREFERRED_LLM_MODEL, {}).get(qid, 1)

# Finalizing results using only the preferred model (kbtg)
voted_preds = {q['id']: get_final_answer(q['id'], hybrid_preds_multi) for q in questions[:N_QUESTIONS]}

In [26]:
voted_preds

{1: 5,
 2: 7,
 3: 2,
 4: 6,
 5: 6,
 6: 8,
 7: 1,
 8: 4,
 9: 4,
 10: 7,
 11: 4,
 12: 1,
 13: 2,
 14: 3,
 15: 7,
 16: 1,
 17: 8,
 18: 5,
 19: 2,
 20: 2,
 21: 3,
 22: 8,
 23: 9,
 24: 9,
 25: 5,
 26: 6,
 27: 2,
 28: 7,
 29: 4,
 30: 3,
 31: 4,
 32: 2,
 33: 8,
 34: 5,
 35: 3,
 36: 2,
 37: 8,
 38: 6,
 39: 4,
 40: 8,
 41: 7,
 42: 2,
 43: 4,
 44: 1,
 45: 2,
 46: 1,
 47: 9,
 48: 8,
 49: 6,
 50: 5,
 51: 7,
 52: 4,
 53: 9,
 54: 9,
 55: 9,
 56: 9,
 57: 9,
 58: 9,
 59: 9,
 60: 10,
 61: 9,
 62: 9,
 63: 9,
 64: 5,
 65: 3,
 66: 7,
 67: 6,
 68: 1,
 69: 2,
 70: 8,
 71: 4,
 72: 7,
 73: 6,
 74: 5,
 75: 6,
 76: 1,
 77: 1,
 78: 3,
 79: 7,
 80: 1,
 81: 9,
 82: 9,
 83: 4,
 84: 5,
 85: 1,
 86: 1,
 87: 1,
 88: 2,
 89: 4,
 90: 6,
 91: 4,
 92: 4,
 93: 5,
 94: 4,
 95: 4,
 96: 5,
 97: 7,
 98: 5,
 99: 4,
 100: 1}

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [29]:
with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        # Use voted prediction if available, else default to 1
        writer.writerow([qid, voted_preds.get(qid, 1)])

print(f"submission.csv written successfully! ({len(questions)} rows)")

submission.csv written successfully! (100 rows)


---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**